Revisar monedas y cobertura-------------------------------------------------------------------------------------------------------------------------------------

In [0]:
SELECT moneda, COUNT(*) AS contratos, ROUND(SUM(monto_contrato)) AS monto
FROM compras_rd.gold.fact_contratos
GROUP BY moneda
ORDER BY contratos DESC;

KPIs generales (para las tarjetas del dashboard)----------------------------------------------------------------------------------------------------------------

In [0]:
SELECT
  COUNT(*)                                  AS contratos,
  COUNT(DISTINCT proveedor_id)              AS proveedores,
  COUNT(DISTINCT institucion_id)            AS instituciones,
  ROUND(SUM(monto_contrato))                AS monto_total,
  ROUND(AVG(monto_contrato))                AS ticket_promedio,
  ROUND(percentile_approx(monto_contrato, 0.5)) AS ticket_mediano
FROM compras_rd.gold.fact_contratos
WHERE NOT flag_monto_invalido
  AND moneda = 'DOP';

Evolución mensual y estacionalidad----------------------------------------------------------------------------------------------------------------------------

In [0]:
-- Evolución mensual y estacionalidad
SELECT
  anio, mes, num_contratos,
  ROUND(monto_total)                                   AS monto_total,
  ROUND(monto_total / LAG(monto_total) OVER (ORDER BY anio, mes) - 1, 3) AS var_vs_mes_anterior,
  ROUND(monto_total / SUM(monto_total) OVER (PARTITION BY anio), 3)      AS peso_en_el_anio
FROM compras_rd.gold.agg_gasto_mensual
WHERE moneda = 'DOP'
ORDER BY anio, mes;

In [0]:

SELECT
  make_date(df.anio, df.mes, 1)  AS mes_fecha,
  di.institucion,
  ROUND(SUM(fc.monto_contrato))  AS monto_total
FROM compras_rd.gold.fact_contratos fc
LEFT JOIN compras_rd.gold.dim_fecha df       ON fc.fecha_key = df.fecha_key
LEFT JOIN compras_rd.gold.dim_institucion di ON fc.institucion_id = di.institucion_id
WHERE NOT fc.flag_monto_invalido AND fc.moneda = 'DOP'
GROUP BY make_date(df.anio, df.mes, 1), di.institucion

Instituciones con mayor gasto

In [0]:
SELECT
  institucion, num_contratos,
  ROUND(monto_total)                                              AS monto_total,
  ROUND(monto_total / SUM(monto_total) OVER (), 4)                AS participacion,
  ROUND(SUM(monto_total) OVER (ORDER BY monto_total DESC)
        / SUM(monto_total) OVER (), 4)                            AS participacion_acumulada
FROM compras_rd.gold.agg_gasto_institucion
WHERE moneda = 'DOP'
ORDER BY monto_total DESC
LIMIT 15;

Concentración de proveedores (Pareto)

In [0]:
 --¿Cuántos proveedores explican el 80% del monto? 
WITH v AS (
  SELECT proveedor_id, SUM(monto_contrato) AS m
  FROM compras_rd.gold.fact_contratos
  WHERE NOT flag_monto_invalido AND moneda = 'DOP' AND proveedor_id IS NOT NULL
  GROUP BY proveedor_id
),
r AS (
  SELECT m,
         m / SUM(m) OVER ()                                AS part,
         SUM(m) OVER (ORDER BY m DESC) / SUM(m) OVER ()    AS acum
  FROM v
)
SELECT
  COUNT(*)                                                       AS total_proveedores,
  SUM(CASE WHEN acum - part < 0.8 THEN 1 ELSE 0 END)             AS proveedores_para_80pct,
  ROUND(100.0 * SUM(CASE WHEN acum - part < 0.8 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_de_proveedores
FROM r;

In [0]:

--Top 10 proveedores


SELECT
  p.nombre, p.escala, r.n_contratos,
  ROUND(r.m)                                          AS monto_total,
  ROUND(r.m / SUM(r.m) OVER (), 4)                    AS participacion
FROM (
  SELECT proveedor_id, COUNT(*) AS n_contratos, SUM(monto_contrato) AS m
  FROM compras_rd.gold.fact_contratos
  WHERE NOT flag_monto_invalido AND moneda = 'DOP' AND proveedor_id IS NOT NULL
  GROUP BY proveedor_id
) r
LEFT JOIN compras_rd.gold.dim_proveedor p ON r.proveedor_id = p.proveedor_id
ORDER BY r.m DESC
LIMIT 10;

Modalidades de compra

In [0]:
SELECT
  COALESCE(m.modalidad, 'sin dato')        AS modalidad,
  COALESCE(m.modalidad_detalle, 'sin dato') AS detalle,
  COUNT(*)                                  AS contratos,
  ROUND(SUM(fc.monto_contrato))             AS monto_total,
  ROUND(SUM(fc.monto_contrato) / SUM(SUM(fc.monto_contrato)) OVER (), 4) AS pct_monto,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 4)                             AS pct_contratos
FROM compras_rd.gold.fact_contratos fc
LEFT JOIN compras_rd.gold.dim_modalidad m ON fc.modalidad_key = m.modalidad_key
WHERE NOT fc.flag_monto_invalido AND fc.moneda = 'DOP'
GROUP BY m.modalidad, m.modalidad_detalle
ORDER BY monto_total DESC;

Tiempo de adjudicación por modalidad

In [0]:
SELECT
  COALESCE(m.modalidad, 'sin dato') AS modalidad,
  COUNT(*)                                            AS procesos,
  ROUND(AVG(fp.dias_hasta_adjudicacion), 1)           AS dias_promedio,
  percentile_approx(fp.dias_hasta_adjudicacion, 0.5)  AS dias_mediana,
  percentile_approx(fp.dias_hasta_adjudicacion, 0.9)  AS dias_p90
FROM compras_rd.gold.fact_procesos fp
LEFT JOIN compras_rd.gold.dim_modalidad m ON fp.modalidad_key = m.modalidad_key
WHERE fp.dias_hasta_adjudicacion >= 0
GROUP BY m.modalidad
HAVING COUNT(*) >= 30
ORDER BY dias_mediana DESC;

In [0]:
SELECT
  SUM(CASE WHEN dias_hasta_adjudicacion < 0 THEN 1 ELSE 0 END) AS tiempos_negativos,
  SUM(CASE WHEN dias_hasta_adjudicacion IS NULL THEN 1 ELSE 0 END) AS sin_fechas,
  COUNT(*) AS total
FROM compras_rd.gold.fact_procesos;

MIPYMES y mujeres empresarias

In [0]:
SELECT
  COALESCE(p.escala, 'sin dato')            AS escala,
  COUNT(*)                                  AS contratos,
  ROUND(SUM(fc.monto_contrato))             AS monto_total,
  ROUND(SUM(fc.monto_contrato) / SUM(SUM(fc.monto_contrato)) OVER (), 4) AS pct_monto
FROM compras_rd.gold.fact_contratos fc
LEFT JOIN compras_rd.gold.dim_proveedor p ON fc.proveedor_id = p.proveedor_id
WHERE NOT fc.flag_monto_invalido AND fc.moneda = 'DOP'
GROUP BY p.escala
ORDER BY monto_total DESC;

In [0]:
SELECT
  COALESCE(p.genero, 'sin dato')            AS genero,
  COUNT(*)                                  AS contratos,
  ROUND(SUM(fc.monto_contrato))             AS monto_total,
  ROUND(SUM(fc.monto_contrato) / SUM(SUM(fc.monto_contrato)) OVER (), 4) AS pct_monto
FROM compras_rd.gold.fact_contratos fc
LEFT JOIN compras_rd.gold.dim_proveedor p ON fc.proveedor_id = p.proveedor_id
WHERE NOT fc.flag_monto_invalido AND fc.moneda = 'DOP'
GROUP BY p.genero
ORDER BY monto_total DESC;

Monto estimado frente a monto contratado

In [0]:
SELECT
  COALESCE(m.modalidad, 'sin dato')                    AS modalidad,
  COUNT(*)                                             AS procesos,
  ROUND(SUM(fp.monto_estimado))                        AS estimado,
  ROUND(SUM(fp.monto_contratado))                      AS contratado,
  ROUND(SUM(fp.monto_contratado) / SUM(fp.monto_estimado) - 1, 4) AS desvio_total,
  ROUND(percentile_approx(fp.monto_contratado / fp.monto_estimado - 1, 0.5), 4) AS desvio_mediano
FROM compras_rd.gold.fact_procesos fp
LEFT JOIN compras_rd.gold.dim_modalidad m ON fp.modalidad_key = m.modalidad_key
WHERE fp.moneda = 'DOP'
  AND fp.monto_estimado > 0
  AND fp.monto_contratado > 0
GROUP BY m.modalidad
HAVING COUNT(*) >= 30
ORDER BY procesos DESC;

Indicadores de alerta

In [0]:
--instituciones con muchos procesos adjudicados con un solo oferente
SELECT
  di.institucion,
  COUNT(*)                                                     AS procesos_con_contrato,
  SUM(CASE WHEN fp.num_oferentes = 1 THEN 1 ELSE 0 END)        AS un_solo_oferente,
  ROUND(AVG(CASE WHEN fp.num_oferentes = 1 THEN 1.0 ELSE 0.0 END), 3) AS pct_un_oferente
FROM compras_rd.gold.fact_procesos fp
LEFT JOIN compras_rd.gold.dim_institucion di ON fp.institucion_id = di.institucion_id
WHERE fp.num_contratos > 0 AND fp.num_oferentes IS NOT NULL
GROUP BY di.institucion
HAVING COUNT(*) >= 30
ORDER BY pct_un_oferente DESC
LIMIT 15;

In [0]:
--proveedor dominante dentro de una institucion 
WITH x AS (
  SELECT
    fc.institucion_id, fc.proveedor_id,
    COUNT(*)                   AS n,
    SUM(fc.monto_contrato)     AS m
  FROM compras_rd.gold.fact_contratos fc
  WHERE NOT fc.flag_monto_invalido AND fc.moneda = 'DOP'
    AND fc.proveedor_id IS NOT NULL
  GROUP BY fc.institucion_id, fc.proveedor_id
),
y AS (
  SELECT *,
         m / SUM(m) OVER (PARTITION BY institucion_id)       AS participacion,
         SUM(n) OVER (PARTITION BY institucion_id)           AS contratos_institucion,
         ROW_NUMBER() OVER (PARTITION BY institucion_id ORDER BY m DESC) AS rk
  FROM x
)
SELECT
  di.institucion, p.nombre AS proveedor_dominante,
  y.contratos_institucion, ROUND(y.m) AS monto, ROUND(y.participacion, 3) AS participacion
FROM y
LEFT JOIN compras_rd.gold.dim_institucion di ON y.institucion_id = di.institucion_id
LEFT JOIN compras_rd.gold.dim_proveedor p ON y.proveedor_id = p.proveedor_id
WHERE y.rk = 1 AND y.participacion >= 0.5 AND y.contratos_institucion >= 20
ORDER BY y.participacion DESC
LIMIT 20;